# Loan Approval Prediction
Logistic Regression, Random Forest, and KNN.

## Step 1 — Load Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

df = pd.read_csv("clean_loan_dataset.csv")
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,-0.445244,8.274536
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,-0.912749,0.153994
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,0.280113,-0.126321
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.315175,-0.669033
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.284688,-0.284975


## Step 2 — Features & Target

In [2]:
X = df.drop(columns=["Approved"])
y = df["Approved"]
X.shape, y.shape

((100, 8), (100,))

## Step 3 — Train/Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((80, 8), (20, 8))

## Step 4 — Train Logistic Regression & Random Forest

In [4]:
lr = LogisticRegression(max_iter=1000, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Step 5 — Third Classifier: KNN

I chose **K-Nearest Neighbors (KNN)** as the third classifier. KNN predicts the label of a new data point by looking at the K closest training examples and taking a majority vote. It works well on small, clean datasets like this one where the decision boundary between approved and rejected applicants may not be linear.

**Source:** scikit-learn documentation — [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [5]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


## Step 6 — Evaluate All Three Models

In [6]:
def print_clf_metrics(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1)
    rec  = recall_score(y_true, y_pred, pos_label=1)
    f1   = f1_score(y_true, y_pred, pos_label=1)
    print(f"{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (positive = Approved=1)")
    print(f"  Recall   : {rec:.3f}  (positive = Approved=1)")
    print(f"  F1-Score : {f1:.3f}  (positive = Approved=1)")
    print()

def print_confmat(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: Approved", "Actual: Rejected"],
        columns=["Pred: Approved", "Pred: Rejected"]
    )
    print(f"{name} — Confusion Matrix:")
    print(cm_df)
    print()

lr_pred  = lr.predict(X_test)
rf_pred  = rf.predict(X_test)
knn_pred = knn.predict(X_test)

print_clf_metrics("Logistic Regression", y_test, lr_pred)
print_clf_metrics("Random Forest",       y_test, rf_pred)
print_clf_metrics("KNN",                 y_test, knn_pred)

Logistic Regression Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Random Forest Performance:
  Accuracy : 0.650
  Precision: 0.714  (positive = Approved=1)
  Recall   : 0.769  (positive = Approved=1)
  F1-Score : 0.741  (positive = Approved=1)

KNN Performance:
  Accuracy : 0.650
  Precision: 0.688  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.759  (positive = Approved=1)



In [7]:
print_confmat("Logistic Regression", y_test, lr_pred)
print_confmat("Random Forest",       y_test, rf_pred)
print_confmat("KNN",                 y_test, knn_pred)

Logistic Regression — Confusion Matrix:
                  Pred: Approved  Pred: Rejected
Actual: Approved              11               2
Actual: Rejected               4               3

Random Forest — Confusion Matrix:
                  Pred: Approved  Pred: Rejected
Actual: Approved              10               3
Actual: Rejected               4               3

KNN — Confusion Matrix:
                  Pred: Approved  Pred: Rejected
Actual: Approved              11               2
Actual: Rejected               5               2



## Step 7 — Sanity Check

In [8]:
def label_str(v):
    return "Approved (1)" if v == 1 else "Rejected (0)"

i = 2
x_one = X_test.iloc[[i]]
y_true = y_test.iloc[i]

print(f"Actual label        : {label_str(y_true)}")
print(f"Logistic Regression : {label_str(int(lr.predict(x_one)[0]))}")
print(f"Random Forest       : {label_str(int(rf.predict(x_one)[0]))}")
print(f"KNN                 : {label_str(int(knn.predict(x_one)[0]))}")

Actual label        : Approved (1)
Logistic Regression : Approved (1)
Random Forest       : Approved (1)
KNN                 : Approved (1)
